# Covariance Calibration Playground

A self-contained Torch estimator example. It builds synthetic constant-velocity data, starts with poor process and measurement covariances, and learns diagonal Q/R by backpropagating through the full Kalman-filter rollout.

Run this notebook on CPU or CUDA. CUDA is faster, but the example is intentionally small.


In [ ]:
import math
import torch
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


## Synthetic Trajectory

The state is `[px, py, vx, vy]`. The estimator receives noisy acceleration as a control input and noisy position measurements.


In [ ]:
def make_dataset(T=300, dt=0.02, accel_noise=0.08, pos_noise=0.12, seed=3):
    gen = torch.Generator().manual_seed(seed)
    t = torch.arange(T, dtype=torch.float64) * dt
    accel_true = torch.stack([
        0.8 * torch.sin(1.2 * t),
        0.5 * torch.cos(0.8 * t + 0.3),
    ], dim=-1)

    x = torch.zeros(T, 4, dtype=torch.float64)
    x[0] = torch.tensor([0.0, 0.0, 0.4, -0.2], dtype=torch.float64)
    for k in range(T - 1):
        x[k + 1, 0:2] = x[k, 0:2] + x[k, 2:4] * dt + 0.5 * accel_true[k] * dt * dt
        x[k + 1, 2:4] = x[k, 2:4] + accel_true[k] * dt

    accel_meas = accel_true + accel_noise * torch.randn(T, 2, generator=gen)
    pos_meas = x[:, 0:2] + pos_noise * torch.randn(T, 2, generator=gen)
    return x.to(device), accel_meas.to(device), pos_meas.to(device), dt

x_gt, accel_meas, pos_meas, dt = make_dataset()
print('trajectory shape:', x_gt.shape)


## Differentiable Estimator

`raw_q` and `raw_r` are trainable diagonal covariance parameters. The rollout keeps the whole computation graph, including propagation, Kalman gain, and Joseph covariance update.


In [ ]:
class CovarianceLearner(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Bad initialization: huge process covariance, tiny measurement covariance.
        self.raw_q = torch.nn.Parameter(torch.tensor([1.5, 1.5, 1.5, 1.5], device=device))
        self.raw_r = torch.nn.Parameter(torch.tensor([-6.0, -6.0], device=device))

    def covariances(self):
        q = torch.nn.functional.softplus(self.raw_q) + 1e-9
        r = torch.nn.functional.softplus(self.raw_r) + 1e-9
        return torch.diag(q), torch.diag(r)

    def forward(self, accel, pos, dt):
        T = pos.shape[0]
        F = torch.eye(4, device=device)
        F[0, 2] = dt
        F[1, 3] = dt
        B = torch.tensor([
            [0.5 * dt * dt, 0.0],
            [0.0, 0.5 * dt * dt],
            [dt, 0.0],
            [0.0, dt],
        ], device=device)
        H = torch.tensor([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]], device=device)
        I = torch.eye(4, device=device)
        Q, R = self.covariances()

        x = torch.zeros(4, device=device)
        x[0:2] = pos[0]
        P = torch.diag(torch.tensor([10.0, 10.0, 1.0, 1.0], device=device))
        states = []
        for k in range(T):
            if k > 0:
                x = F @ x + B @ accel[k - 1]
                P = F @ P @ F.T + Q

            y = pos[k] - H @ x
            S = H @ P @ H.T + R
            K = torch.linalg.solve(S.T, (P @ H.T).T).T
            x = x + K @ y
            IKH = I - K @ H
            P = IKH @ P @ IKH.T + K @ R @ K.T
            states.append(x)
        return torch.stack(states)


def trajectory_loss(x_hat, x_gt):
    pos = ((x_hat[:, 0:2] - x_gt[:, 0:2]) ** 2).sum(-1).mean()
    vel = 0.2 * ((x_hat[:, 2:4] - x_gt[:, 2:4]) ** 2).sum(-1).mean()
    return pos + vel


## Train Q/R

The initial covariances make the filter overreact to noisy position measurements. Adam adjusts Q/R directly from trajectory error.


In [ ]:
model = CovarianceLearner().to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-2)
losses = []
for step in range(500):
    opt.zero_grad(set_to_none=True)
    x_hat = model(accel_meas, pos_meas, dt)
    loss = trajectory_loss(x_hat, x_gt)
    loss.backward()
    opt.step()
    losses.append(float(loss.detach().cpu()))
    if step % 100 == 0 or step == 499:
        Q, R = model.covariances()
        print(f'step {step:03d} loss={losses[-1]:.4e} diagQ={Q.diag().detach().cpu().numpy()} diagR={R.diag().detach().cpu().numpy()}')


## Inspect the Result


In [ ]:
with torch.no_grad():
    learned = model(accel_meas, pos_meas, dt)
    bad = CovarianceLearner().to(device)(accel_meas, pos_meas, dt)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].semilogy(losses)
axes[0].set_title('training loss')
axes[0].set_xlabel('step')
axes[0].grid(True, alpha=0.3)
axes[1].plot(x_gt[:, 0].cpu(), x_gt[:, 1].cpu(), 'k-', label='ground truth')
axes[1].plot(pos_meas[:, 0].cpu(), pos_meas[:, 1].cpu(), '.', ms=2, alpha=0.25, label='measurements')
axes[1].plot(bad[:, 0].detach().cpu(), bad[:, 1].detach().cpu(), '--', label='bad init')
axes[1].plot(learned[:, 0].detach().cpu(), learned[:, 1].detach().cpu(), '-', label='trained')
axes[1].axis('equal')
axes[1].set_title('trajectory')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## What to Try

- Change the bad initialization in `CovarianceLearner`.
- Increase measurement noise in `make_dataset`.
- Replace diagonal Q/R with Cholesky-parameterized full SPD matrices.
- Train on multiple synthetic trajectories by summing their rollout losses.
